# ROS dataset audit and source checks

Question: which recordings support small, explainable investigations, and what do the selected warnings actually establish?

Run after `python scripts/prepare_investigations.py` and `python scripts/audit_dataset_inventory.py` from the repository environment. This notebook reads originals and completed evidence; it does not download, mutate datasets, tune thresholds or delete files. Missing Docker-volume access remains a scope limitation.


In [1]:
import json
from pathlib import Path

import numpy as np
import polars as pl

from ros_telemetry_analytics.discovery import discover_bags
from ros_telemetry_analytics.investigations import load_bundle, specs
from ros_telemetry_analytics.reader import open_bag

ROOT = Path.cwd() if (Path.cwd() / "configs/investigations.yaml").exists() else Path.cwd().parent
OUTPUT = ROOT / "data/investigations"
assert (ROOT / "configs/investigations.yaml").exists(), (
    "Start in the repository or examples directory"
)

## Admission and lineage
Each scorecard is bound to a source digest, a source stat fingerprint, and a recipe signature. Local digests identify copies; they are not proof of upstream authenticity. A ready bundle indicates successful extraction and sampled reconciliation, not that every robot signal is healthy or every message type has payload support.


In [2]:
bundles = {name: load_bundle(ROOT, OUTPUT, name) for name in specs(ROOT)}
summary = []
for name, (_directory, meta) in bundles.items():
    assert sum(row["message_count"] for row in meta["coverage"]) == meta["message_count"]
    summary.append(
        {
            "dataset": name,
            "seconds": meta["duration_s"],
            "messages": meta["message_count"],
            "errors": meta["extraction_errors"],
            "previews_checked": meta["verified_previews"],
            "previews": meta["preview_count"],
            "status": meta["status"],
        }
    )
print(pl.DataFrame(summary))

shape: (7, 7)
┌─────────────────────────┬────────────┬──────────┬────────┬──────────────────┬──────────┬────────┐
│ dataset                 ┆ seconds    ┆ messages ┆ errors ┆ previews_checked ┆ previews ┆ status │
│ ---                     ┆ ---        ┆ ---      ┆ ---    ┆ ---              ┆ ---      ┆ ---    │
│ str                     ┆ f64        ┆ i64      ┆ i64    ┆ i64              ┆ i64      ┆ str    │
╞═════════════════════════╪════════════╪══════════╪════════╪══════════════════╪══════════╪════════╡
│ tum_rgbd_freiburg1_xyz  ┆ 30.428861  ┆ 25626    ┆ 0      ┆ 12               ┆ 12       ┆ ready  │
│ tum_vi_room4_512        ┆ 111.404953 ┆ 39743    ┆ 0      ┆ 18               ┆ 18       ┆ ready  │
│ lilocbench_dynamics_0   ┆ 159.978066 ┆ 30577    ┆ 0      ┆ 18               ┆ 18       ┆ ready  │
│ lilocbench_static_0     ┆ 598.794301 ┆ 110812   ┆ 0      ┆ 12               ┆ 12       ┆ ready  │
│ lilocbench_lt_changes_0 ┆ 435.988775 ┆ 85117    ┆ 0      ┆ 12               ┆ 12    

## Independent image check
Re-read the exact cam0 source message near the selected image interval and compute its sampled mean directly from 16-bit pixels, independently of the analyzer helper. TUM VI encodes exposure nanoseconds in image `frame_id`; it is not a camera coordinate frame. Source: [TUM VI paper, format section](https://cvg.cit.tum.de/_media/spezial/bib/schubert2018vidataset.pdf).


In [3]:
directory, meta = bundles["tum_vi_room4_512"]
previews = json.loads((directory / "previews.json").read_text())
sample = min(
    (p for p in previews if p["topic"] == "/cam0/image_raw"), key=lambda p: abs(p["t"] - 102.903)
)
(source,) = discover_bags([ROOT / specs(ROOT)["tum_vi_room4_512"]["input"]])
with open_bag(source) as reader:
    for connection, stamp, raw in reader.messages():
        if connection.topic == sample["topic"] and str(stamp) == sample["timestamp_ns"]:
            message = reader.deserialize(raw, connection.msgtype)
            byte_rows = np.asarray(message.data, dtype=np.uint8).reshape(
                message.height, message.step
            )
            active = np.ascontiguousarray(byte_rows[:, : message.width * 2])
            order = ">" if message.is_bigendian else "<"
            pixels = np.frombuffer(active, dtype=order + "u2").reshape(
                message.height, message.width
            )
            stride = max(1, max(message.height, message.width) // 256)
            measured = float((pixels.astype(np.float32) / 257)[::stride, ::stride].mean())
            assert abs(measured - sample["mean_intensity"]) < 1e-6
            print(
                {
                    "t": sample["t"],
                    "independent_mean": measured,
                    "stored_mean": sample["mean_intensity"],
                    "exposure_ns": message.header.frame_id,
                }
            )
            break
    else:
        raise AssertionError("Exact source sample missing")

{'t': 102.902854667, 'independent_mean': 15.672110557556152, 'stored_mean': 15.672110557556152, 'exposure_ns': '818526'}


## Reference coverage versus delivery
Use integer nanoseconds for differences. A reference-pose gap is an evaluation limitation. Sorted recorded timestamps do not prove arrival order or packet loss.


In [4]:
index = pl.read_parquet(directory / meta["analysis_path"] / "message_index.parquet")
profile = []
for group in index.partition_by("topic", maintain_order=True):
    stamps = group["timestamp_ns"].sort().to_numpy()
    gaps = np.diff(stamps)
    largest = int(gaps.argmax())
    profile.append(
        {
            "topic": group["topic"][0],
            "max_gap_ms": float(gaps.max() / 1e6),
            "start_s": (int(stamps[largest]) - int(meta["origin_ns"])) / 1e9,
            "end_s": (int(stamps[largest + 1]) - int(meta["origin_ns"])) / 1e9,
        }
    )
print(pl.DataFrame(profile))

shape: (4, 4)
┌────────────────────────────┬────────────┬────────────┬────────────┐
│ topic                      ┆ max_gap_ms ┆ start_s    ┆ end_s      │
│ ---                        ┆ ---        ┆ ---        ┆ ---        │
│ str                        ┆ f64        ┆ f64        ┆ f64        │
╞════════════════════════════╪════════════╪════════════╪════════════╡
│ /cam0/image_raw            ┆ 51.727965  ┆ 102.952856 ┆ 103.004584 │
│ /cam1/image_raw            ┆ 51.727965  ┆ 102.952856 ┆ 103.004584 │
│ /imu0                      ┆ 5.036      ┆ 31.089929  ┆ 31.094965  │
│ /vrpn_client/raw_transform ┆ 483.334    ┆ 45.210764  ┆ 45.694098  │
└────────────────────────────┴────────────┴────────────┴────────────┘


## Scan spot check and motion interpretation
Independently count finite ranges inside the message's stated limits. This establishes structural scan validity, not localization accuracy. Commands are unstamped Twist messages; odometry twist uses its child frame. Axis comparison needs platform conventions and cannot prove wheel slip.


In [5]:
directory, meta = bundles["lilocbench_dynamics_0"]
previews = json.loads((directory / "previews.json").read_text())
sample = min(
    (p for p in previews if p["topic"] == "/laser_scan_front/scan"),
    key=lambda p: abs(p["t"] - 65.82),
)
(source,) = discover_bags([ROOT / specs(ROOT)["lilocbench_dynamics_0"]["input"]])
with open_bag(source) as reader:
    for connection, stamp, raw in reader.messages():
        if connection.topic == sample["topic"] and str(stamp) == sample["timestamp_ns"]:
            message = reader.deserialize(raw, connection.msgtype)
            values = [float(v) for v in message.ranges]
            valid = sum(
                np.isfinite(v) and message.range_min <= v <= message.range_max for v in values
            )
            fraction = valid / len(values)
            assert abs(fraction - sample["valid_range_fraction"]) < 1e-12
            print(
                {
                    "t": sample["t"],
                    "valid_beams": valid,
                    "beams": len(values),
                    "valid_fraction": fraction,
                }
            )
            break
    else:
        raise AssertionError("Exact source scan missing")

{'t': 65.814872139, 'valid_beams': 783, 'beams': 811, 'valid_fraction': 0.9654747225647349}


## Collection scope and decisions
Keep seven real recordings active, the synthetic warehouse fixture for controlled timing tests, TUHH for simulation-based localization evaluation, NVIDIA for compatibility and the four legacy fixtures for optional parser regressions. Hide deferred downloads from the main replay selector. Exact duplicate cleanup preserves the canonical extraction and archive. Inspect the audit receipt for which paths were removed.


In [6]:
audit_root = ROOT / "data/evaluations/dataset-audit"
audit_id = json.loads((audit_root / "latest.json").read_text())["audit_id"]
audit = json.loads((audit_root / audit_id / "inventory.json").read_text())
print(
    {
        "audit_id": audit_id,
        "files": len(audit["physical_files"]),
        "logical_bags": len(audit["logical_bags"]),
        "host_bytes": sum(f["bytes"] for f in audit["physical_files"]),
        "docker_volumes": audit["docker_volumes"]["status"],
    }
)
assert not audit["discovery_errors"]

{'audit_id': 'a04ce50102044110aed1e953e9e2f9fb', 'files': 77, 'logical_bags': 12, 'host_bytes': 10596815779, 'docker_volumes': 'unavailable'}


## What this establishes
These recordings can support the three reviewed questions. Neither warning counts nor successful extraction certify a physically faulty or fault-free robot. Static/dynamic/changed-environment routes are not matched causal experiments. Public LILocBench reference trajectories are acquired but not aligned or used by detectors. Docker-volume inventory and a full streaming-stack smoke test remain separate checks when Docker is available.
